### Filter Verification V2

In [ ]:
import os
import datetime as dt

from dotenv import load_dotenv
from pathlib import Path

import pywt
import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm

from scipy.stats import spearmanr
from scipy.stats import pearsonr
from scipy.stats import linregress
from scipy.stats import fisher_exact
from scipy.signal import butter
from scipy.signal import lfilter
from sklearn.metrics import roc_auc_score
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.filters.hp_filter import hpfilter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score
)
from sklearn.feature_selection import (
    mutual_info_regression
)
from pykalman import KalmanFilter
from tqdm import tqdm

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.request_api_v2 import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown
from Modules.compute_fibonacci_extension import ComputeFibonacciExtension
from Modules.fibonacci_retracement import FibonacciRetracement
from Modules.fibonacci_extension import FibonacciExtension

In [2]:
# FRED API key for accessing Federal Reserve Economic Data
FRED_API_KEY = os.getenv("FRED_API_KEY")
# Output directory for storing downloaded data
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
fred_url = "https://api.stlouisfed.org/fred/series/observations"
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()
# Fibonacciエクステンションを計算するクラスのインスタンスを作成
fibonacci_extension = ComputeFibonacciExtension()
# Fibonacciエクステンションをプロットするクラスのインスタンスを作成
fibonacci_extension_v2 = FibonacciExtension() 
# Fibonacciリトレースメントを計算するクラスのインスタンスを作成
fibonacci_retracement = FibonacciRetracement()

In [3]:
# Date range for fetching API s
start = '1950-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
frequency = "d"
aggregation_method = "avg"

In [4]:
def delete_column(df):
    df = df.drop(columns=['realtime_start', 'realtime_end', 'interest_rate_type'])
    return df

#### S&P500(均等加重)時系列を取得

In [5]:
spxew_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="^SPXEW",
    market=None,
    series_name="SPXEW"
))
spxew_df.tail()

取得件数: 4990


,value
date,
2026-10-26,8772.370117
2026-10-27,8802.639648
2026-10-28,8809.419922
2026-10-29,8806.370117
2026-10-30,8888.769531


#### S&P500(時価総額加重平均)時系列を取得

In [6]:
sp500_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="^GSPC",
    market=None,
    series_name="GSPC"
))
sp500_df.tail()

取得件数: 19360


,value
date,
2026-10-26,7639.700195
2026-10-27,7699.470215
2026-10-28,7735.560059
2026-10-29,7699.609863
2026-10-30,7631.069824


#### Nasdaq100(均等加重)時系列を取得

In [7]:
ndxe_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="^NDXE",
    market=None,
    series_name="NDXE"
))
ndxe_df.tail()

取得件数: 5151


,value
date,
2026-10-26,9917.309570
2026-10-27,10010.910156
2026-10-28,10064.759766
2026-10-29,10038.530273
2026-10-30,9927.026367


#### Nasdaq100(時価総額加重平均)時系列を取得

In [8]:
qqq_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="^IXIC",
    market=None,
    series_name="IXIC"
))
qqq_df.tail()

取得件数: 14052


,value
date,
2026-10-26,26026.869141
2026-10-27,26286.310547
2026-10-28,26541.609375
2026-10-29,26201.820312
2026-10-30,25592.486328


#### iShares iBoxx $ High Yield Corporate Bond ETF

In [9]:
hyg_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="HYG",
    market=None,
    series_name="HYG"
))
hyg_df.tail()

取得件数: 4922


,value
date,
2026-10-26,77.080002
2026-10-27,77.300003
2026-10-28,77.709999
2026-10-29,77.480003
2026-10-30,77.440002


#### iShares 7-10 Year Treasury Bond ETF (IEF)

In [10]:
ief_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="IEF",
    market=None,
    series_name="IEF"
))
ief_df.tail()

取得件数: 6104


,value
date,
2026-10-26,91.160004
2026-10-27,91.059998
2026-10-28,91.059998
2026-10-29,91.110001
2026-10-30,91.610001


#### Federal Funds Effective Rate (DFF)

In [11]:
dff_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DFF": "federal_funds_effective_rate"},
    frequency=frequency,
    aggregation_method=aggregation_method,
)["federal_funds_effective_rate"]
dff_df["date"] = pd.to_datetime(dff_df["date"])
dff_df = dff_df.set_index("date")
dff_df = delete_column(dff_df)
dff_df.head()

,value
date,
1954-07-01,1.13
1954-07-02,1.25
1954-07-03,1.25
1954-07-04,1.25
1954-07-05,0.88


#### Market Yield on U.S. Treasury Securities at 2-Year Constant Maturity, Quoted on an Investment Basis (DGS2)

In [12]:
dgs2_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DGS2": "two_year_treasury_yield"},
    frequency=frequency,
    aggregation_method=aggregation_method,
)["two_year_treasury_yield"]
dgs2_df["date"] = pd.to_datetime(dgs2_df["date"])
dgs2_df = dgs2_df.set_index("date")
dgs2_df = delete_column(dgs2_df)
dgs2_df.head()

,value
date,
1976-06-01,7.26
1976-06-02,7.23
1976-06-03,7.22
1976-06-04,7.12
1976-06-07,7.09


#### Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity, Quoted on an Investment Basis (DGS10)

In [13]:
dgs10_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DGS10": "ten_year_treasury_yield"},
    frequency=frequency,
    aggregation_method=aggregation_method,
)["ten_year_treasury_yield"]
dgs10_df["date"] = pd.to_datetime(dgs10_df["date"])
dgs10_df = dgs10_df.set_index("date")
dgs10_df = delete_column(dgs10_df)
dgs10_df.head()

,value
date,
1962-01-02,4.06
1962-01-03,4.03
1962-01-04,3.99
1962-01-05,4.02
1962-01-08,4.03


#### U.S. Dollars to Australian Dollar Spot Exchange Rate (DEXUSAL)

In [14]:
dxy_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DEXUSAL": "DXY"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["DXY"]
dxy_df["date"] = pd.to_datetime(dxy_df["date"])
dxy_df = dxy_df.set_index("date")
dxy_df = delete_column(dxy_df)
dxy_df.tail()

,value
date,
2026-10-19,0.6981
2026-10-20,0.7004
2026-10-21,0.7026
2026-10-22,0.7020
2026-10-23,0.7037


#### Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity, Quoted on an Investment Basis, Inflation-Indexed (DFII10)

In [15]:
real_yield_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DFII10": "real_10y"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["real_10y"]
real_yield_df["date"] = pd.to_datetime(real_yield_df["date"])
real_yield_df = real_yield_df.set_index("date")
real_yield_df = delete_column(real_yield_df)
real_yield_df.tail()

,value
date,
2026-10-23,2.66
2026-10-26,2.66
2026-10-27,2.64
2026-10-28,2.67
2026-10-29,2.63


#### ICE BofA US High Yield Index Option-Adjusted Spread (BAMLH0A0HYM2)

In [16]:
credit_spread_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"BAMLH0A0HYM2": "credit_spread"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["credit_spread"]
credit_spread_df["date"] = pd.to_datetime(credit_spread_df["date"])
credit_spread_df = credit_spread_df.set_index("date")
credit_spread_df = delete_column(credit_spread_df)
credit_spread_df.tail()

,value
date,
2026-10-23,3.38
2026-10-26,3.30
2026-10-27,3.18
2026-10-28,3.14
2026-10-29,3.26


#### ICE BofA AAA US Corporate Index Option-Adjusted Spread (BAMLC0A1CAAA)

In [17]:
aaa_oas_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"BAMLC0A1CAAA": "AAA_OAS"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["AAA_OAS"]
aaa_oas_df["date"] = pd.to_datetime(aaa_oas_df["date"])
aaa_oas_df = aaa_oas_df.set_index("date")
aaa_oas_df = delete_column(aaa_oas_df)
aaa_oas_df.tail()

,value
date,
2026-10-23,0.44
2026-10-26,0.43
2026-10-27,0.42
2026-10-28,0.41
2026-10-29,0.46


#### ICE BofA BBB US Corporate Index Option-Adjusted Spread (BAMLC0A4CBBB)

In [18]:
bbb_oas_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"BAMLC0A4CBBB": "BBB_OAS"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["BBB_OAS"]
bbb_oas_df["date"] = pd.to_datetime(bbb_oas_df["date"])
bbb_oas_df = bbb_oas_df.set_index("date")
bbb_oas_df = delete_column(bbb_oas_df)
bbb_oas_df.tail()

,value
date,
2026-10-23,1.13
2026-10-26,1.12
2026-10-27,1.09
2026-10-28,1.07
2026-10-29,1.10


#### Federal Funds Effective Rate (FEDFUNDS)

In [19]:
ffer_df = request_api.get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"FEDFUNDS": "ff_rate"},
    frequency='m',
    aggregation_method=aggregation_method
)["ff_rate"]
ffer_df["date"] = pd.to_datetime(ffer_df["date"])
ffer_df = ffer_df.set_index("date")
ffer_df = delete_column(ffer_df)
ffer_df.tail()

,value
date,
2026-05-01,3.63
2026-06-01,3.63
2026-07-01,3.63
2026-08-01,3.63
2026-09-01,3.75


#### データ統合

In [20]:
# ============================================================
# データ統合
# ============================================================
# 各データを共通フォーマットへ変換
df_list = [
    spxew_df.rename(columns={'value': "SPXEW"}),
    sp500_df.rename(columns={'value': "GSPC"}),
    ndxe_df.rename(columns={'value': "NDXE"}),
    qqq_df.rename(columns={'value': "IXIC"}),
    hyg_df.rename(columns={'value': "HYG"}),
    ief_df.rename(columns={'value': "IEF"}),
    dff_df.rename(columns={'value': "DFF"}),
    dgs2_df.rename(columns={'value': "DGS2"}),
    dgs10_df.rename(columns={'value': "DGS10"}),
    dxy_df.rename(columns={'value': "DXY"}),
    real_yield_df.rename(columns={'value': "DFII10"}),
    credit_spread_df.rename(columns={'value': "BAMLH0A0HYM2"}),
    aaa_oas_df.rename(columns={'value': "BAMLC0A1CAAA"}),
    bbb_oas_df.rename(columns={'value': "BAMLC0A4CBBB"}),
    ffer_df.rename(columns={'value': "FEDFUNDS"}),
]
# 日付で外部結合
market_df = pd.concat(df_list, axis=1, join="outer")
# 日付順にソート
market_df = market_df.sort_index()
# 重複日付除去
market_df = market_df[~market_df.index.duplicated(keep="first")]
# 欠損補完
market_df = market_df.ffill()
# 全列欠損行は削除
market_df = market_df.dropna(how="all")
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"Filter_Verification_V2_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = market_df.copy()
df.to_csv(output_path)
# 確認
print("=" * 60)
print("Market Data Shape")
print(market_df.shape)
print("\nColumns")
print(market_df.columns.tolist())
print("\nHead")
display(market_df.head())
print("\nTail")
display(market_df.tail())

Market Data Shape
(27574, 15)

Columns
['SPXEW', 'GSPC', 'NDXE', 'IXIC', 'HYG', 'IEF', 'DFF', 'DGS2', 'DGS10', 'DXY', 'DFII10', 'BAMLH0A0HYM2', 'BAMLC0A1CAAA', 'BAMLC0A4CBBB', 'FEDFUNDS']

Head


/tmp/ipykernel_124/1566513405.py:23: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  market_df = pd.concat(df_list, axis=1, join="outer")


,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,DFII10,BAMLH0A0HYM2,BAMLC0A1CAAA,BAMLC0A4CBBB,FEDFUNDS
date,,,,,,,,,,,,,,,
1949-11-17,NaN,15.960000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949-11-18,NaN,16.139999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949-11-21,NaN,16.070000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949-11-22,NaN,16.150000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949-11-23,NaN,16.190001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Tail


,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,DFII10,BAMLH0A0HYM2,BAMLC0A1CAAA,BAMLC0A4CBBB,FEDFUNDS
date,,,,,,,,,,,,,,,
2026-10-26,8772.370117,7639.700195,9917.309570,26026.869141,77.080002,91.160004,3.88,4.68,5.00,0.7037,2.66,3.30,0.43,1.12,3.75
2026-10-27,8802.639648,7699.470215,10010.910156,26286.310547,77.300003,91.059998,3.88,4.70,4.99,0.7037,2.64,3.18,0.42,1.09,3.75
2026-10-28,8809.419922,7735.560059,10064.759766,26541.609375,77.709999,91.059998,3.88,4.67,5.05,0.7037,2.67,3.14,0.41,1.07,3.75
2026-10-29,8806.370117,7699.609863,10038.530273,26201.820312,77.480003,91.110001,3.88,4.61,4.98,0.7037,2.63,3.26,0.46,1.10,3.75
2026-10-30,8888.769531,7631.069824,9927.026367,25592.486328,77.440002,91.610001,3.88,4.61,4.98,0.7037,2.63,3.26,0.46,1.10,3.75


#### Filter Verification V2

Phase 1
Rolling HP Validation

Phase 2
Causal Kalman Validation

Objective:
Verify whether the strong explanatory power
observed in HP Cycle and Kalman targets
is robust and free from look-ahead bias.

#### 2. Create Research Dataset

Create target returns and explanatory variables.

In [21]:
# 研究用データフレームを作成
research_df = market_df.copy()
research_df.index = pd.to_datetime(research_df.index)
# 対象の株価指数の対数リターンを計算
for target in ["GSPC", "SPXEW"]:
    # 対数リターンを計算
    research_df[f"{target}_LOG_RET"] = np.log(research_df[target]).diff()
# 10年国債利回りの変化を計算
research_df["DGS10_CHANGE"] = (
    research_df["DGS10"].diff()
)
# 2年国債利回りの変化を計算
research_df["DFII10_CHANGE"] = (
    research_df["DFII10"].diff()
)
# DXYの変化を計算
research_df["DXY_CHANGE"] = (
    research_df["DXY"].pct_change()
)
# ハイイールド債のOASの変化を計算
research_df["HY_OAS_CHANGE"] = (
    research_df["BAMLH0A0HYM2"].diff()
)
# BBB債のOASの変化を計算
research_df["BBB_OAS_CHANGE"] = (
    research_df["BAMLC0A4CBBB"].diff()
)
# HYGの対数リターンを計算
research_df["HYG_LOG_RET"] = (
    np.log(
        research_df["HYG"]
    ).diff()
)
# IEFの対数リターンを計算
research_df["IEF_LOG_RET"] = (
    np.log(
        research_df["IEF"]
    ).diff()
)
# 欠損値を削除
research_df = research_df.dropna()
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"Filter_Verification_V2_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# 確認
research_df.head()

,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,...,FEDFUNDS,GSPC_LOG_RET,SPXEW_LOG_RET,DGS10_CHANGE,DFII10_CHANGE,DXY_CHANGE,HY_OAS_CHANGE,BBB_OAS_CHANGE,HYG_LOG_RET,IEF_LOG_RET
date,,,,,,,,,,,,,,,,,,,,,
2023-11-01,5514.319824,4201.270020,6028.169922,12887.059570,60.863802,80.364675,5.33,4.95,4.77,0.6363,...,5.33,0.007152,0.008523,-0.11,-0.10,0.004896,0.05,0.01,0.003564,0.000219
2023-11-02,5532.330078,4268.259766,6154.319824,13230.490234,61.747931,81.674185,5.33,4.98,4.67,0.6422,...,5.33,0.015819,0.003261,-0.10,-0.09,0.009272,-0.32,-0.04,0.014422,0.016163
2023-11-03,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,5.33,0.015338,0.021584,-0.10,-0.09,0.013236,-0.11,-0.01,0.011121,0.010705
2023-11-04,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,5.33,0.000000,0.000000,0.00,0.00,0.000000,0.00,0.00,0.000000,0.000000
2023-11-05,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,5.33,0.000000,0.000000,0.00,0.00,0.000000,0.00,0.00,0.000000,0.000000


#### 3. Rolling HP Filter

Remove look-ahead bias.

HP Cycle is calculated only
using data available at each time.

In [23]:
# 対象指数の対数収益率に対するローリングHPフィルター
MIN_WINDOW = 252
for target in ["GSPC", "SPXEW"]:
    # 対象指数ごとにローリングHPフィルターを適用
    series = research_df[f"{target}_LOG_RET"]
    # ローリングHPフィルターの結果を格納する配列を初期化
    rolling_cycle = np.full(len(series), np.nan)
    # ローリングウィンドウの開始位置から順にHPフィルターを適用
    for i in tqdm(range(MIN_WINDOW, len(series))):
        # 現在のウィンドウを取得
        window = series.iloc[:i]
        # HPフィルターを適用
        cycle, trend = hpfilter(window, lamb=129600)
        # 現在のウィンドウの最後のサイクル成分を格納
        rolling_cycle[i] = (cycle.iloc[-1])
    # ローリングHPフィルターの結果をデータフレームに格納
    research_df[f"{target}_ROLLING_HP"] = rolling_cycle
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"3_Rolling_HP_Filter_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# 確認
research_df.head()

100%|██████████| 843/843 [00:00<00:00, 3044.30it/s]


,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,...,SPXEW_LOG_RET,DGS10_CHANGE,DFII10_CHANGE,DXY_CHANGE,HY_OAS_CHANGE,BBB_OAS_CHANGE,HYG_LOG_RET,IEF_LOG_RET,GSPC_ROLLING_HP,SPXEW_ROLLING_HP
date,,,,,,,,,,,,,,,,,,,,,
2023-11-01,5514.319824,4201.270020,6028.169922,12887.059570,60.863802,80.364675,5.33,4.95,4.77,0.6363,...,0.008523,-0.11,-0.10,0.004896,0.05,0.01,0.003564,0.000219,NaN,NaN
2023-11-02,5532.330078,4268.259766,6154.319824,13230.490234,61.747931,81.674185,5.33,4.98,4.67,0.6422,...,0.003261,-0.10,-0.09,0.009272,-0.32,-0.04,0.014422,0.016163,NaN,NaN
2023-11-03,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,0.021584,-0.10,-0.09,0.013236,-0.11,-0.01,0.011121,0.010705,NaN,NaN
2023-11-04,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,0.000000,0.00,0.00,0.000000,0.00,0.00,0.000000,0.000000,NaN,NaN
2023-11-05,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,0.000000,0.00,0.00,0.000000,0.00,0.00,0.000000,0.000000,NaN,NaN


#### 4. Full Sample HP Filter

Original HP filter
potentially containing future information.

In [24]:
# 対象指数の対数収益率に対するHPフィルターの適用
for target in ["GSPC", "SPXEW"]:
    # 対数収益率にHPフィルターを適用
    cycle, trend = hpfilter(
        research_df[
            f"{target}_LOG_RET"
        ],
        lamb=129600
    )
    # サイクル成分とトレンド成分をデータフレームに格納
    research_df[f"{target}_HP_CYCLE"] = cycle
    # トレンド成分をデータフレームに格納
    research_df[f"{target}_HP_TREND"] = trend
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"4_Full_Sample_HP_Filter_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# 確認
research_df.head()

,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,...,HY_OAS_CHANGE,BBB_OAS_CHANGE,HYG_LOG_RET,IEF_LOG_RET,GSPC_ROLLING_HP,SPXEW_ROLLING_HP,GSPC_HP_CYCLE,GSPC_HP_TREND,SPXEW_HP_CYCLE,SPXEW_HP_TREND
date,,,,,,,,,,,,,,,,,,,,,
2023-11-01,5514.319824,4201.270020,6028.169922,12887.059570,60.863802,80.364675,5.33,4.95,4.77,0.6363,...,0.05,0.01,0.003564,0.000219,NaN,NaN,0.002709,0.004443,0.004288,0.004236
2023-11-02,5532.330078,4268.259766,6154.319824,13230.490234,61.747931,81.674185,5.33,4.98,4.67,0.6422,...,-0.32,-0.04,0.014422,0.016163,NaN,NaN,0.011476,0.004343,-0.000902,0.004163
2023-11-03,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,-0.11,-0.01,0.011121,0.010705,NaN,NaN,0.011095,0.004243,0.017495,0.004090
2023-11-04,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,0.00,0.00,0.000000,0.000000,NaN,NaN,-0.004142,0.004142,-0.004017,0.004017
2023-11-05,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,0.00,0.00,0.000000,0.000000,NaN,NaN,-0.004043,0.004043,-0.003944,0.003944


#### 5. Causal Kalman Filter

One-step recursive estimation.

 No future information allowed.

In [25]:
# 対象指数の対数収益率に対するカルマンフィルター（因果型）の適用
for target in ["GSPC", "SPXEW"]:
    # 対数収益率を取得し、欠損値を0で埋めて配列に変換
    observations = (
        research_df[f"{target}_LOG_RET"]
        .fillna(0)
        .values
    )
    # カルマンフィルターの初期化
    kf = KalmanFilter(
        transition_matrices=[1],
        observation_matrices=[1],
        transition_covariance=0.001,
        observation_covariance=0.01
    )
    # フィルター後の状態平均を格納するリストを初期化
    state_means = []
    # 初期状態の設定
    state_mean = 0
    state_cov = 1
    # 観測値に対してカルマンフィルターを適用
    for obs in observations:
        state_mean, state_cov = (
            kf.filter_update(
                state_mean,
                state_cov,
                observation=obs
            )
        )
        # フィルター後の状態平均をリストに追加
        state_means.append(float(state_mean))
    # カルマンフィルターの結果をデータフレームに格納
    research_df[f"{target}_KALMAN_CAUSAL"] = state_means
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"5_Causal_Kalman_Filter_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# 確認
research_df.head()

,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,...,HYG_LOG_RET,IEF_LOG_RET,GSPC_ROLLING_HP,SPXEW_ROLLING_HP,GSPC_HP_CYCLE,GSPC_HP_TREND,SPXEW_HP_CYCLE,SPXEW_HP_TREND,GSPC_KALMAN_CAUSAL,SPXEW_KALMAN_CAUSAL
date,,,,,,,,,,,,,,,,,,,,,
2023-11-01,5514.319824,4201.270020,6028.169922,12887.059570,60.863802,80.364675,5.33,4.95,4.77,0.6363,...,0.003564,0.000219,NaN,NaN,0.002709,0.004443,0.004288,0.004236,0.007081,0.008439
2023-11-02,5532.330078,4268.259766,6154.319824,13230.490234,61.747931,81.674185,5.33,4.98,4.67,0.6422,...,0.014422,0.016163,NaN,NaN,0.011476,0.004343,-0.000902,0.004163,0.011639,0.005738
2023-11-03,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,0.011121,0.010705,NaN,NaN,0.011095,0.004243,0.017495,0.004090,0.013057,0.011812
2023-11-04,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,0.000000,0.000000,NaN,NaN,-0.004142,0.004142,-0.004017,0.004017,0.008802,0.007963
2023-11-05,5653.040039,4334.229980,6255.629883,13362.849609,62.438450,82.553176,5.33,4.83,4.57,0.6507,...,0.000000,0.000000,NaN,NaN,-0.004043,0.004043,-0.003944,0.003944,0.006173,0.005585


#### 6. Candidate Targets

In [27]:
target_columns = [
    "GSPC_HP_CYCLE",
    "SPXEW_HP_CYCLE",

    "GSPC_ROLLING_HP",
    "SPXEW_ROLLING_HP",
    "GSPC_KALMAN_CAUSAL",
    "SPXEW_KALMAN_CAUSAL",
]

#### 7. Pearson Correlation

In [28]:
features = [
    "DGS10_CHANGE",
    "DFII10_CHANGE",
    "DXY_CHANGE",
    "HY_OAS_CHANGE",
    "BBB_OAS_CHANGE",
    "HYG_LOG_RET",
    "IEF_LOG_RET",
]
# 各特徴量とターゲット列のピアソン相関を計算
results = []
for feature in features:
    # 各特徴量に対してターゲット列との相関を計算
    for target in target_columns:
        # 対象の特徴量とターゲット列のデータを抽出し、欠損値を除去
        tmp = (research_df[[feature, target]].dropna())
        # ピアソン相関を計算するために、欠損値を除去したデータを使用
        corr = (tmp[feature].corr(tmp[target]))
        # 計算結果を辞書に追加
        results.append({
            "feature": feature,
            "target": target,
            "pearson": corr,
        })
# ピアソン相関の結果をデータフレームに変換
pearson_df = pd.DataFrame(results)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"7_Pearson_Correlation_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = pearson_df.copy()
df.to_csv(output_path)
# ピアソン相関の結果を降順で表示
pearson_df.sort_values("pearson",ascending=False)

,feature,target,pearson
30,HYG_LOG_RET,GSPC_HP_CYCLE,0.732659
34,HYG_LOG_RET,GSPC_KALMAN_CAUSAL,0.455636
31,HYG_LOG_RET,SPXEW_HP_CYCLE,0.438761
12,DXY_CHANGE,GSPC_HP_CYCLE,0.371257
35,HYG_LOG_RET,SPXEW_KALMAN_CAUSAL,0.284643
13,DXY_CHANGE,SPXEW_HP_CYCLE,0.276540
16,DXY_CHANGE,GSPC_KALMAN_CAUSAL,0.266822
17,DXY_CHANGE,SPXEW_KALMAN_CAUSAL,0.163259
41,IEF_LOG_RET,SPXEW_KALMAN_CAUSAL,0.141879
37,IEF_LOG_RET,SPXEW_HP_CYCLE,0.085568


#### 8. Spearman Correlation

In [29]:
# スピアマン相関の計算
results = []
for feature in features:
    # 各特徴量に対してターゲット列との相関を計算
    for target in target_columns:
        # 対象の特徴量とターゲット列のデータを抽出し、欠損値を除去
        tmp = (
            research_df[[feature, target]]
            .dropna()
        )
        # スピアマン相関係数とp値を計算
        coef, pvalue = spearmanr(
            tmp[feature],
            tmp[target]
        )
        # 計算結果を辞書に追加
        results.append({
            "feature": feature,
            "target": target,
            "spearman": coef,
            "pvalue": pvalue,
        })

# スピアマン相関の結果をデータフレームに変換
spearman_df = pd.DataFrame(results)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"8_Spearman_Correlation_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = spearman_df.copy()
df.to_csv(output_path)
# スピアマン相関の結果を降順で表示
spearman_df.sort_values("spearman",ascending=False)

,feature,target,spearman,pvalue
30,HYG_LOG_RET,GSPC_HP_CYCLE,0.583848,4.789416e-101
34,HYG_LOG_RET,GSPC_KALMAN_CAUSAL,0.328999,4.705619e-29
31,HYG_LOG_RET,SPXEW_HP_CYCLE,0.273595,2.982241e-20
12,DXY_CHANGE,GSPC_HP_CYCLE,0.244694,2.156628e-16
35,HYG_LOG_RET,SPXEW_KALMAN_CAUSAL,0.191047,1.848022e-10
16,DXY_CHANGE,GSPC_KALMAN_CAUSAL,0.173791,7.105933e-09
36,IEF_LOG_RET,GSPC_HP_CYCLE,0.104097,5.603895e-04
13,DXY_CHANGE,SPXEW_HP_CYCLE,0.103112,6.324992e-04
41,IEF_LOG_RET,SPXEW_KALMAN_CAUSAL,0.088754,3.288679e-03
17,DXY_CHANGE,SPXEW_KALMAN_CAUSAL,0.085598,4.590078e-03


#### 9. Mutual Information

In [30]:
# 相互情報量の計算
results = []
for feature in features:
    # 各特徴量に対してターゲットとの相互情報量を計算
    for target in target_columns:
        # 欠損値を除外してデータを準備
        tmp = (research_df[[feature, target]].dropna())
        # 特徴量とターゲットを分割
        X = tmp[[feature]]
        y = tmp[target]
        # 相互情報量を計算
        mi = (mutual_info_regression(X, y, random_state=42)[0])
        # 計算結果を保存
        results.append({
            "feature": feature,
            "target": target,
            "mi": mi,
        })
# 計算結果をデータフレームに変換
mi_df = pd.DataFrame(results)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"9_Mutual_Information_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = mi_df.copy()
df.to_csv(output_path)
# 相互情報量の結果を表示
mi_df.sort_values("mi", ascending=False)

,feature,target,mi
30,HYG_LOG_RET,GSPC_HP_CYCLE,0.440000
31,HYG_LOG_RET,SPXEW_HP_CYCLE,0.299804
37,IEF_LOG_RET,SPXEW_HP_CYCLE,0.293440
18,HY_OAS_CHANGE,GSPC_HP_CYCLE,0.293262
13,DXY_CHANGE,SPXEW_HP_CYCLE,0.263429
1,DGS10_CHANGE,SPXEW_HP_CYCLE,0.260771
12,DXY_CHANGE,GSPC_HP_CYCLE,0.257286
36,IEF_LOG_RET,GSPC_HP_CYCLE,0.243696
6,DFII10_CHANGE,GSPC_HP_CYCLE,0.194276
7,DFII10_CHANGE,SPXEW_HP_CYCLE,0.181414


#### 10. Leakage Comparison

Compare:

HP Cycle
vs
Rolling HP

If explanatory power collapses,
previous results may be caused
by look-ahead bias.

In [31]:
# 各種相関指標を比較するデータフレームを作成
comparison_df = (
    pearson_df
    .merge(spearman_df, on=["feature", "target"])
    .merge(mi_df, on=["feature", "target"])
)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"10_Leakage_Comparison_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = comparison_df.copy()
df.to_csv(output_path)
# MIでソートして上位50件を表示
comparison_df.sort_values("mi", ascending=False).head(50)

,feature,target,pearson,spearman,pvalue,mi
30,HYG_LOG_RET,GSPC_HP_CYCLE,0.732659,0.583848,4.789416e-101,0.440000
31,HYG_LOG_RET,SPXEW_HP_CYCLE,0.438761,0.273595,2.982241e-20,0.299804
37,IEF_LOG_RET,SPXEW_HP_CYCLE,0.085568,0.058440,5.320365e-02,0.293440
18,HY_OAS_CHANGE,GSPC_HP_CYCLE,-0.425387,-0.338673,8.606468e-31,0.293262
13,DXY_CHANGE,SPXEW_HP_CYCLE,0.276540,0.103112,6.324992e-04,0.263429
1,DGS10_CHANGE,SPXEW_HP_CYCLE,-0.038662,-0.033904,2.623094e-01,0.260771
12,DXY_CHANGE,GSPC_HP_CYCLE,0.371257,0.244694,2.156628e-16,0.257286
36,IEF_LOG_RET,GSPC_HP_CYCLE,0.041906,0.104097,5.603895e-04,0.243696
6,DFII10_CHANGE,GSPC_HP_CYCLE,-0.046069,-0.054961,6.906415e-02,0.194276
7,DFII10_CHANGE,SPXEW_HP_CYCLE,-0.054685,-0.030563,3.122845e-01,0.181414


# Filter Verification V2 レポート

## 検証目的

Filter Verification V1では、HP Filter適用後のCycle成分に対して非常に高い説明力が確認された。

しかし、通常のHP Filterは全データ期間を使用してTrendを推定するため、未来情報の混入（Look-Ahead Bias）が疑われた。

本検証では、

```text
Rolling HP Filter
Causal Kalman Filter
```

を導入し、観測された説明力がリーキングによるものかを検証した。 【1-0d302b】【2-ee8557】【3-4de3c3】【4-f10bbe】

---

## 検証手法

### Rolling HP

各時点において過去データのみを利用してHP Cycleを計算し、未来情報を完全に排除した。

### Causal Kalman

逐次更新型カルマンフィルタを使用し、各時点で利用可能な情報のみから状態推定を実施した。

評価指標として

```text
Pearson Correlation
Spearman Correlation
Mutual Information
```

を使用した。 【1-0d302b】【2-ee8557】【3-4de3c3】【4-f10bbe】

---

## 主な結果

### Rolling HP

V1で強い説明力を示した関係の大部分が消失した。

例：

```text
HYG_LOG_RET
→ GSPC_HP_CYCLE

Pearson 0.733
MI      0.440
```

↓

```text
HYG_LOG_RET
→ GSPC_ROLLING_HP

Pearson -0.287
MI       0.022
```

となった。 【1-0d302b】【2-ee8557】【4-f10bbe】

同様に、

```text
HY_OAS_CHANGE
BBB_OAS_CHANGE
```

についても説明力はほぼ消失した。 【1-0d302b】【2-ee8557】

### Causal Kalman

一方でカルマンフィルタでは説明力が一定程度維持された。

代表例：

```text
HYG_LOG_RET
→ GSPC_KALMAN_CAUSAL

Pearson  = 0.456
Spearman = 0.329
MI       = 0.114
```

```text
BBB_OAS_CHANGE
→ GSPC_KALMAN_CAUSAL

Pearson  = -0.381
Spearman = -0.257
MI       = 0.086
```

```text
HY_OAS_CHANGE
→ GSPC_KALMAN_CAUSAL

Pearson  = -0.305
Spearman = -0.201
MI       = 0.077
```

が確認された。 【1-0d302b】【2-ee8557】【3-4de3c3】【4-f10bbe】

---

## 結論

本検証により、

```text
HP Filterで観測された強い説明力の多くは
Look-Ahead Biasによる可能性が高い
```

ことが判明した。 【1-0d302b】【2-ee8557】

一方で、

```text
HYG
HY OAS
BBB OAS
```

といった信用市場変数は、未来情報を使用しないCausal Kalman環境下でも一定の説明力を維持した。 【1-0d302b】【2-ee8557】【3-4de3c3】【4-f10bbe】

現時点では、

```text
信用市場変数に一定の説明力は存在する

ただしV1で観測されたほど強力ではない
```

という結論が最も妥当である。

今後は、

```text
Kalmanベースの分析
Wavelet Approximationの導入
Surrogate Test
Out-of-Sample検証
```

を進め、観測された関係の頑健性を検証する。